A **heap** is a complete binary tree that satisfies one additional constraint: every node is smaller than (or equal to) its children — the **min-heap property**. The root is always the minimum element, accessible in **O(1)**. Insertion and removal both run in **O(log n)**.

Heaps power **priority queues** — data structures that always serve the highest-priority item first, regardless of insertion order. They are the engine behind Dijkstra's shortest-path algorithm, Prim's MST algorithm, A\* search, and any problem that repeatedly asks "what is the current minimum (or maximum)?".

## Heap Property and Array Storage

A **min-heap** satisfies: `heap[parent] ≤ heap[child]` for every node.
A **max-heap** satisfies: `heap[parent] ≥ heap[child]`.

Because a heap is a **complete binary tree**, it can be stored efficiently in a plain array — no pointers needed. Level-order position maps directly to index:

| Relationship | Formula (0-indexed) |
|---|---|
| Parent of i | `(i - 1) // 2` |
| Left child of i | `2*i + 1` |
| Right child of i | `2*i + 2` |

The minimum (for a min-heap) is always at index 0.

![Heap Array Mapping](https://raw.githubusercontent.com/schemabotview/dsa/main/img/heap-array-mapping.svg)

## Insert and Sift Up — O(log n)

1. Append the new value at the end of the array (maintains the complete-tree shape).
2. **Sift up**: compare with its parent. If smaller (min-heap), swap. Repeat until the heap property holds or the new value reaches the root.

At most one swap per level → O(log n).

![Heap Insert Sift Up](https://raw.githubusercontent.com/schemabotview/dsa/main/img/heap-insert-siftup.svg)

## Heap Implementation from Scratch

### Python

In [ ]:
class MinHeap:
    def __init__(self):
        self._data: list[int] = []

    # ── index arithmetic ──────────────────────────────────────
    def _parent(self, i: int) -> int: return (i - 1) // 2
    def _left(self,   i: int) -> int: return 2 * i + 1
    def _right(self,  i: int) -> int: return 2 * i + 2

    def peek(self) -> int:
        """Return minimum without removing — O(1)."""
        if not self._data:
            raise IndexError("heap is empty")
        return self._data[0]

    def push(self, val: int) -> None:
        """Insert val — O(log n)."""
        self._data.append(val)           # step 1: append
        self._sift_up(len(self._data) - 1)

    def pop(self) -> int:
        """Remove and return minimum — O(log n)."""
        if not self._data:
            raise IndexError("heap is empty")
        # Swap root with last element, shrink array, sift root down
        self._data[0], self._data[-1] = self._data[-1], self._data[0]
        minimum = self._data.pop()
        self._sift_down(0)
        return minimum

    def _sift_up(self, i: int) -> None:
        while i > 0:
            p = self._parent(i)
            if self._data[i] < self._data[p]:
                self._data[i], self._data[p] = self._data[p], self._data[i]
                i = p
            else:
                break

    def _sift_down(self, i: int) -> None:
        n = len(self._data)
        while True:
            smallest, l, r = i, self._left(i), self._right(i)
            if l < n and self._data[l] < self._data[smallest]: smallest = l
            if r < n and self._data[r] < self._data[smallest]: smallest = r
            if smallest == i:
                break                    # heap property holds
            self._data[i], self._data[smallest] = self._data[smallest], self._data[i]
            i = smallest

    def __len__(self):  return len(self._data)
    def __repr__(self): return f"MinHeap({self._data})"


h = MinHeap()
for v in [8, 3, 5, 1, 4, 6, 9]:
    h.push(v)

print(h)                      # MinHeap([1, 3, 5, 8, 4, 6, 9])
print(h.peek())               # 1
print([h.pop() for _ in range(len(h))])  # [1, 3, 4, 5, 6, 8, 9] — sorted!

### Kotlin

```kotlin
class MinHeap {
    private val data = mutableListOf<Int>()

    private fun parent(i: Int) = (i - 1) / 2
    private fun left(i: Int)   = 2 * i + 1
    private fun right(i: Int)  = 2 * i + 2

    fun peek(): Int = data.first()
    val size get() = data.size

    fun push(value: Int) {
        data.add(value)
        siftUp(data.lastIndex)
    }

    fun pop(): Int {
        val min = data[0]
        data[0] = data.removeLast()
        if (data.isNotEmpty()) siftDown(0)
        return min
    }

    private fun siftUp(i: Int) {
        var idx = i
        while (idx > 0) {
            val p = parent(idx)
            if (data[idx] < data[p]) { data[idx] = data[p].also { data[p] = data[idx] }; idx = p }
            else break
        }
    }

    private fun siftDown(i: Int) {
        var idx = i
        while (true) {
            var smallest = idx
            val l = left(idx); val r = right(idx)
            if (l < data.size && data[l] < data[smallest]) smallest = l
            if (r < data.size && data[r] < data[smallest]) smallest = r
            if (smallest == idx) break
            data[idx] = data[smallest].also { data[smallest] = data[idx] }
            idx = smallest
        }
    }
}
```

## Python's `heapq` — Built-in Min-Heap

Python provides `heapq` — a module of heap operations on a plain list. It is always a **min-heap**.

### Python

In [ ]:
import heapq

# ── Min-heap ──────────────────────────────────────────────────
nums = [8, 3, 5, 1, 4, 6, 9]
heapq.heapify(nums)              # O(n) in-place — converts list to heap
print(nums)                      # [1, 3, 5, 8, 4, 6, 9]

heapq.heappush(nums, 2)          # O(log n) insert
print(heapq.heappop(nums))       # 1 — O(log n) remove min
print(nums[0])                   # 2 — O(1) peek (access index 0 directly)

# ── Max-heap trick: negate values ────────────────────────────
max_heap: list[int] = []
for v in [8, 3, 5, 1, 4]:
    heapq.heappush(max_heap, -v)  # push negated

print(-heapq.heappop(max_heap))   # 8 — negate on extraction

# ── nsmallest / nlargest — O(n log k) ────────────────────────
data = [10, 3, 7, 1, 8, 5, 2]
print(heapq.nsmallest(3, data))   # [1, 2, 3]
print(heapq.nlargest(3,  data))   # [10, 8, 7]

### Kotlin — `PriorityQueue`

```kotlin
import java.util.PriorityQueue

// Min-heap (default)
val minHeap = PriorityQueue<Int>()
listOf(8, 3, 5, 1, 4).forEach { minHeap.offer(it) }
println(minHeap.peek())   // 1  — O(1)
println(minHeap.poll())   // 1  — O(log n)

// Max-heap — reverse comparator
val maxHeap = PriorityQueue<Int>(compareByDescending { it })
listOf(8, 3, 5, 1, 4).forEach { maxHeap.offer(it) }
println(maxHeap.poll())   // 8

// Custom objects — min-heap by frequency
data class Entry(val word: String, val freq: Int)
val pq = PriorityQueue<Entry>(compareBy { it.freq })
pq.offer(Entry("apple", 5))
pq.offer(Entry("banana", 2))
println(pq.poll().word)   // banana (lowest freq first)
```

## Build Heap — O(n)

Naively inserting n elements one by one takes O(n log n). There is a faster approach: start with the raw array, then call sift-down on every non-leaf node, working **bottom-up** from index `n//2 - 1` down to `0`. This builds a valid heap in **O(n)**.

The O(n) proof: most nodes are near the leaves and require very few swaps. The total work sums to O(n), not O(n log n).

### Python

In [ ]:
def build_heap(arr: list[int]) -> list[int]:
    """Convert arr into a min-heap in-place — O(n)."""
    n = len(arr)

    def sift_down(i: int) -> None:
        while True:
            smallest = i
            l, r = 2*i+1, 2*i+2
            if l < n and arr[l] < arr[smallest]: smallest = l
            if r < n and arr[r] < arr[smallest]: smallest = r
            if smallest == i: break
            arr[i], arr[smallest] = arr[smallest], arr[i]
            i = smallest

    # Start from last non-leaf and sift each down
    for i in range(n // 2 - 1, -1, -1):
        sift_down(i)
    return arr

data = [9, 4, 7, 1, 8, 3, 6, 5, 2]
print(build_heap(data))   # [1, 2, 3, 5, 8, 7, 6, 9, 4]  — valid min-heap

## Canonical Patterns

### Pattern 1 — K Largest Elements

In [ ]:
import heapq

def k_largest(nums: list[int], k: int) -> list[int]:
    """
    O(n log k) — maintain a min-heap of size k.
    The heap's root is the smallest of the k largest seen so far.
    """
    heap: list[int] = []
    for n in nums:
        heapq.heappush(heap, n)
        if len(heap) > k:
            heapq.heappop(heap)   # evict the smallest — not in top-k
    return sorted(heap, reverse=True)

print(k_largest([3, 1, 5, 12, 2, 11, 8], k=3))   # [12, 11, 8]

### Pattern 2 — K Most Frequent Elements

### Python

In [ ]:
from collections import Counter
import heapq

def top_k_frequent(nums: list[int], k: int) -> list[int]:
    """
    O(n log k) — count frequencies, then use a min-heap of size k
    keyed by frequency.
    """
    freq = Counter(nums)
    # heap entries: (frequency, value) — min-heap by frequency
    heap: list[tuple[int, int]] = []
    for val, cnt in freq.items():
        heapq.heappush(heap, (cnt, val))
        if len(heap) > k:
            heapq.heappop(heap)   # remove least frequent
    return [val for _, val in heap]

print(top_k_frequent([1,1,1,2,2,3], k=2))   # [1, 2]

### Kotlin — K Most Frequent

```kotlin
import java.util.PriorityQueue

fun topKFrequent(nums: IntArray, k: Int): List<Int> {
    val freq = nums.groupingBy { it }.eachCount()
    // min-heap by frequency — evict the least frequent when size > k
    val pq = PriorityQueue<Map.Entry<Int, Int>>(compareBy { it.value })
    for (entry in freq.entries) {
        pq.offer(entry)
        if (pq.size > k) pq.poll()
    }
    return pq.map { it.key }
}
```

### Pattern 3 — Merge K Sorted Lists

### Python

In [ ]:
import heapq

def merge_k_sorted(lists: list[list[int]]) -> list[int]:
    """
    O(n log k) where n = total elements, k = number of lists.
    Push the first element of each list into the heap as (value, list_idx, element_idx).
    """
    heap: list[tuple[int, int, int]] = []
    for i, lst in enumerate(lists):
        if lst:
            heapq.heappush(heap, (lst[0], i, 0))

    result = []
    while heap:
        val, i, j = heapq.heappop(heap)
        result.append(val)
        if j + 1 < len(lists[i]):
            heapq.heappush(heap, (lists[i][j + 1], i, j + 1))
    return result

lists = [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
print(merge_k_sorted(lists))   # [1, 2, 3, 4, 5, 6, 7, 8, 9]

### Kotlin — Merge K Sorted Lists

```kotlin
import java.util.PriorityQueue

fun mergeKSorted(lists: List<List<Int>>): List<Int> {
    // Triple: (value, listIndex, elementIndex)
    val pq = PriorityQueue<Triple<Int,Int,Int>>(compareBy { it.first })
    lists.forEachIndexed { i, list -> if (list.isNotEmpty()) pq.offer(Triple(list[0], i, 0)) }

    val result = mutableListOf<Int>()
    while (pq.isNotEmpty()) {
        val (v, i, j) = pq.poll()
        result.add(v)
        if (j + 1 < lists[i].size) pq.offer(Triple(lists[i][j + 1], i, j + 1))
    }
    return result
}
```

## Heap Sort — O(n log n)

Heap sort uses the heap to sort in-place:
1. Build a max-heap from the array — O(n).
2. Repeatedly swap the root (maximum) with the last element, shrink the heap boundary by 1, and sift down — O(log n) per step.

Result: sorted in ascending order, **O(n log n)** time, **O(1)** extra space.

### Python

In [ ]:
def heap_sort(arr: list[int]) -> list[int]:
    n = len(arr)

    def sift_down_max(i: int, size: int) -> None:
        while True:
            largest, l, r = i, 2*i+1, 2*i+2
            if l < size and arr[l] > arr[largest]: largest = l
            if r < size and arr[r] > arr[largest]: largest = r
            if largest == i: break
            arr[i], arr[largest] = arr[largest], arr[i]
            i = largest

    # 1. Build max-heap
    for i in range(n // 2 - 1, -1, -1):
        sift_down_max(i, n)

    # 2. Extract max one by one
    for end in range(n - 1, 0, -1):
        arr[0], arr[end] = arr[end], arr[0]   # move current max to sorted position
        sift_down_max(0, end)                  # restore heap on reduced array

    return arr

print(heap_sort([5, 3, 8, 1, 9, 2, 7, 4, 6]))   # [1, 2, 3, 4, 5, 6, 7, 8, 9]

## Complexity Summary

| Operation | Time | Notes |
|---|---|---|
| `peek` (min/max) | **O(1)** | Always at index 0 |
| `push` | **O(log n)** | Sift up at most h levels |
| `pop` | **O(log n)** | Sift down at most h levels |
| `heapify` (build) | **O(n)** | Bottom-up sift-down |
| `heap sort` | **O(n log n)** | O(1) extra space |
| k largest/smallest | **O(n log k)** | Fixed-size heap of k |
| Merge k sorted lists | **O(n log k)** | n total elements, k lists |

## Key Takeaways

- A **min-heap** guarantees `parent ≤ children` at every node. The minimum is always at the root — O(1) access.
- Heaps are stored as **arrays** using index arithmetic: parent at `(i-1)//2`, left child at `2i+1`, right at `2i+2`. No pointers needed.
- **Push**: append + sift up — O(log n). **Pop**: swap root with last + sift down — O(log n). **Peek**: read index 0 — O(1).
- **Build heap** bottom-up in O(n) — faster than n individual inserts (O(n log n)).
- Python's `heapq` is always a min-heap. Simulate a max-heap by negating values.
- Kotlin's `PriorityQueue` is a min-heap by default; pass a reverse comparator for a max-heap.
- The three canonical heap patterns: **k largest/smallest** (fixed-size heap), **top-k frequent** (heap on frequency), **merge k sorted** (heap on front elements).
- **Heap sort** builds a max-heap in O(n) then extracts the max n times — O(n log n) total, O(1) extra space.